# 02b: Feature Engineering — Approach 2: Loyalty Prediction

**Problem reframing:** Instead of predicting purchase within a time window, we predict whether a first-time customer becomes a repeat buyer based on their initial order characteristics.

- **Target:** Did the customer ever place more than 1 order? (binary)
- **Features:** Derived from the customer's first order only
- **Positive rate:** 3.12% (2,997 repeat buyers out of 96,096 customers)

In [1]:
import pandas as pd
import numpy as np

data_dir = '../datasets'

customers_df = pd.read_csv(f'{data_dir}/olist_customers_dataset.csv')
orders_df = pd.read_csv(f'{data_dir}/olist_orders_dataset.csv')
order_items_df = pd.read_csv(f'{data_dir}/olist_order_items_dataset.csv')
order_payments_df = pd.read_csv(f'{data_dir}/olist_order_payments_dataset.csv')
order_reviews_df = pd.read_csv(f'{data_dir}/olist_order_reviews_dataset.csv')
products_df = pd.read_csv(f'{data_dir}/olist_products_dataset.csv')
product_translation_df = pd.read_csv(f'{data_dir}/product_category_name_translation.csv')

date_cols = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
             'order_delivered_customer_date', 'order_estimated_delivery_date']
for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

order_reviews_df['review_creation_date'] = pd.to_datetime(order_reviews_df['review_creation_date'])

In [2]:
## Step 1: Identify each customer's first order and build targets

In [3]:
# Join orders with customers to get customer_unique_id
orders_customers = orders_df.merge(
    customers_df[['customer_id', 'customer_unique_id']], 
    on='customer_id', how='left'
)

# Count total orders per customer (for target)
customer_order_counts = orders_customers.groupby('customer_unique_id')['order_id'].nunique().reset_index()
customer_order_counts.columns = ['customer_unique_id', 'lifetime_orders']

# Target: repeat buyer = 1 if lifetime_orders > 1
customer_order_counts['y_loyalty'] = (customer_order_counts['lifetime_orders'] > 1).astype(int)

# Find first order per customer
first_orders = orders_customers.sort_values('order_purchase_timestamp').groupby('customer_unique_id').first().reset_index()

# Merge target onto first orders
first_orders = first_orders.merge(customer_order_counts[['customer_unique_id', 'lifetime_orders', 'y_loyalty']], 
                                   on='customer_unique_id', how='left')

print(f"Total customers: {len(first_orders)}")
print(f"Repeat buyers: {first_orders['y_loyalty'].sum()} ({first_orders['y_loyalty'].mean()*100:.2f}%)")
print(f"First orders shape: {first_orders.shape}")

Total customers: 96096
Repeat buyers: 2997 (3.12%)
First orders shape: (96096, 11)


## Step 2: Build first-order features

For each customer, we extract features ONLY from their first order, what they bought, how they paid, how much they spent, shipping cost, product category, review behavior, and delivery experience.

In [4]:
# === PAYMENT FEATURES from first order ===
first_order_ids = first_orders[['order_id', 'customer_unique_id']]

payment_features = first_order_ids.merge(order_payments_df, on='order_id', how='left')

payment_agg = payment_features.groupby('customer_unique_id').agg(
    first_payment_value=('payment_value', 'sum'),
    first_payment_installments=('payment_installments', 'max'),
    first_num_payment_methods=('payment_type', 'nunique'),
    first_payment_sequential=('payment_sequential', 'sum'),
).reset_index()

# Payment type split
payment_type_split = payment_features.groupby(['customer_unique_id', 'payment_type'])['payment_value'].sum().unstack(fill_value=0)
payment_type_split.columns = ['first_pymt_' + col for col in payment_type_split.columns]
payment_type_split = payment_type_split.reset_index()

payment_all = payment_agg.merge(payment_type_split, on='customer_unique_id', how='left')
print(f"Payment features: {payment_all.shape}")

# === ITEM FEATURES from first order ===
item_features = first_order_ids.merge(order_items_df, on='order_id', how='left')

item_agg = item_features.groupby('customer_unique_id').agg(
    first_num_items=('order_item_id', 'count'),
    first_num_sellers=('seller_id', 'nunique'),
    first_total_price=('price', 'sum'),
    first_avg_price=('price', 'mean'),
    first_total_freight=('freight_value', 'sum'),
    first_avg_freight=('freight_value', 'mean'),
).reset_index()
print(f"Item features: {item_agg.shape}")

# === REVIEW FEATURES from first order ===
review_features = first_order_ids.merge(order_reviews_df, on='order_id', how='left')
review_features['review_comment_title'] = review_features['review_comment_title'].fillna('')
review_features['review_comment_message'] = review_features['review_comment_message'].fillna('')
review_features['title_length'] = review_features['review_comment_title'].str.len()
review_features['body_length'] = review_features['review_comment_message'].str.len()

review_agg = review_features.groupby('customer_unique_id').agg(
    first_review_score=('review_score', 'mean'),
    first_review_title_length=('title_length', 'mean'),
    first_review_body_length=('body_length', 'mean'),
).reset_index()
print(f"Review features: {review_agg.shape}")

# === PRODUCT CATEGORY from first order ===
items_with_products = item_features.merge(products_df[['product_id', 'product_category_name']], on='product_id', how='left')
items_with_english = items_with_products.merge(product_translation_df, on='product_category_name', how='left')

def get_favorite(series):
    modes = series.mode()
    if len(modes) > 0:
        return modes.iloc[0]
    return "unknown"

category_agg = items_with_english.groupby('customer_unique_id').agg(
    first_product_category=('product_category_name_english', get_favorite)
).reset_index()
print(f"Category features: {category_agg.shape}")

# === DELIVERY EXPERIENCE from first order ===
first_orders['delivery_days'] = (first_orders['order_delivered_customer_date'] - first_orders['order_purchase_timestamp']).dt.days
first_orders['estimated_delivery_days'] = (first_orders['order_estimated_delivery_date'] - first_orders['order_purchase_timestamp']).dt.days
first_orders['delivery_delta'] = first_orders['estimated_delivery_days'] - first_orders['delivery_days']  # positive = early, negative = late

delivery_features = first_orders[['customer_unique_id', 'delivery_days', 'estimated_delivery_days', 'delivery_delta']].copy()
print(f"Delivery features: {delivery_features.shape}")

# === ORDER METADATA ===
first_orders['purchase_hour'] = first_orders['order_purchase_timestamp'].dt.hour
first_orders['purchase_dayofweek'] = first_orders['order_purchase_timestamp'].dt.dayofweek

order_meta = first_orders[['customer_unique_id', 'order_status', 'purchase_hour', 'purchase_dayofweek']].copy()
print(f"Order metadata: {order_meta.shape}")

Payment features: (96096, 10)
Item features: (96096, 7)
Review features: (96096, 4)
Category features: (96096, 2)
Delivery features: (96096, 4)
Order metadata: (96096, 4)


In [5]:
## Step 3: Merge all features + compute ratio features

In [6]:
# Merge all feature groups
user_df = first_orders[['customer_unique_id', 'y_loyalty', 'lifetime_orders']].copy()

user_df = user_df.merge(payment_all, on='customer_unique_id', how='left') \
                 .merge(item_agg, on='customer_unique_id', how='left') \
                 .merge(review_agg, on='customer_unique_id', how='left') \
                 .merge(category_agg, on='customer_unique_id', how='left') \
                 .merge(delivery_features, on='customer_unique_id', how='left') \
                 .merge(order_meta, on='customer_unique_id', how='left')

# === RATIO FEATURES (the reference repo found these critical) ===
# Value-to-volume: how much they spend per item
user_df['price_per_item'] = user_df['first_total_price'] / user_df['first_num_items']
# Freight ratio: freight as % of total order value
user_df['freight_ratio'] = user_df['first_total_freight'] / (user_df['first_total_price'] + 0.01)
# Installment intensity: installments relative to order value
user_df['installment_per_value'] = user_df['first_payment_installments'] / (user_df['first_payment_value'] + 0.01)

print(f"Shape after merge + ratios: {user_df.shape}")
print(f"\nNull counts:")
print(user_df.isnull().sum()[user_df.isnull().sum() > 0])

Shape after merge + ratios: (96096, 31)

Null counts:
first_payment_installments       1
first_pymt_boleto                1
first_pymt_credit_card           1
first_pymt_debit_card            1
first_pymt_not_defined           1
first_pymt_voucher               1
first_avg_price                708
first_avg_freight              708
first_review_score             735
delivery_days                 2740
delivery_delta                2740
price_per_item                 708
installment_per_value            1
dtype: int64


### Null handling

| Feature(s) | Nulls | Root Cause | Strategy |
|---|---|---|---|
| Payment columns | 1 each | Order with no payment record | Fill 0 |
| first_avg_price, first_avg_freight, price_per_item | 708 | Orders with no items in source | Fill median |
| first_review_score | 735 | Customer never left a review | Fill median |
| delivery_days, delivery_delta | 2,740 | Orders never delivered (canceled/processing) | Fill median |
| installment_per_value | 1 | Derived from null payment | Fill 0 |

In [7]:
# Payment nulls → 0
pymt_cols = ['first_payment_installments', 'first_pymt_boleto', 'first_pymt_credit_card', 
             'first_pymt_debit_card', 'first_pymt_not_defined', 'first_pymt_voucher', 'installment_per_value']
user_df[pymt_cols] = user_df[pymt_cols].fillna(0)

# Item nulls → median
item_null_cols = ['first_avg_price', 'first_avg_freight', 'price_per_item']
for col in item_null_cols:
    user_df[col] = user_df[col].fillna(user_df[col].median())

# Review nulls → median
user_df['first_review_score'] = user_df['first_review_score'].fillna(user_df['first_review_score'].median())

# Delivery nulls → median (not max this time — these are duration features, not recency)
user_df['delivery_days'] = user_df['delivery_days'].fillna(user_df['delivery_days'].median())
user_df['delivery_delta'] = user_df['delivery_delta'].fillna(user_df['delivery_delta'].median())

print(f"Nulls remaining: {user_df.isnull().sum().sum()}")
print(f"Final shape: {user_df.shape}")

Nulls remaining: 0
Final shape: (96096, 31)


In [8]:
## Step 4: Compute order value target for repeat buyers

In [9]:
# For repeat buyers, compute average order value EXCLUDING their first order
all_order_items = orders_customers.merge(order_items_df[['order_id', 'price']], on='order_id', how='inner')

# Sum prices per order
order_values = all_order_items.groupby(['order_id', 'customer_unique_id'])['price'].sum().reset_index()
order_values.columns = ['order_id', 'customer_unique_id', 'order_total']

# Add order timestamps and rank orders per customer
order_values = order_values.merge(
    orders_customers[['order_id', 'customer_unique_id', 'order_purchase_timestamp']].drop_duplicates(),
    on=['order_id', 'customer_unique_id'], how='left'
)
order_values['order_rank'] = order_values.groupby('customer_unique_id')['order_purchase_timestamp'].rank()

# Future order value = average of all orders AFTER the first
future_orders = order_values[order_values['order_rank'] > 1]
customer_future_value = future_orders.groupby('customer_unique_id')['order_total'].mean().reset_index()
customer_future_value.columns = ['customer_unique_id', 'y_future_order_value']

# Merge into user_df
user_df = user_df.merge(customer_future_value, on='customer_unique_id', how='left')
user_df['y_future_order_value'] = user_df['y_future_order_value'].fillna(0)
user_df['y_future_order_value_log'] = np.log1p(user_df['y_future_order_value'])

print(f"Shape: {user_df.shape}")
print(f"\nFuture order value for repeat buyers:")
print(user_df[user_df['y_loyalty'] == 1]['y_future_order_value'].describe())

Shape: (96096, 33)

Future order value for repeat buyers:
count    2997.000000
mean      120.863971
std       152.006658
min         0.000000
25%        43.500000
50%        80.000000
75%       143.873333
max      2589.000000
Name: y_future_order_value, dtype: float64


In [10]:
# Encode order_status
user_df['is_delivered'] = (user_df['order_status'] == 'delivered').astype(int)
user_df = user_df.drop(columns=['order_status'])

# Encode product category — top 10 + other
top_10 = user_df['first_product_category'].value_counts().head(10).index
user_df['first_product_category'] = user_df['first_product_category'].apply(
    lambda x: x if x in top_10 else 'other'
)
user_df = pd.get_dummies(user_df, columns=['first_product_category'], dtype=int)

# Check correlations > 0.7
numeric_cols = user_df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['y_loyalty', 'y_future_order_value', 'y_future_order_value_log', 'lifetime_orders']]

corr = user_df[numeric_cols].corr()
high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            high_corr.append((corr.columns[i], corr.columns[j], round(corr.iloc[i, j], 3)))

print("Highly correlated pairs (|r| > 0.7):")
for pair in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
    print(f"  {pair[0]:30s} <-> {pair[1]:30s} r={pair[2]}")

Highly correlated pairs (|r| > 0.7):
  first_avg_price                <-> price_per_item                 r=1.0
  first_payment_value            <-> first_total_price              r=0.985
  first_total_price              <-> first_avg_price                r=0.933
  first_total_price              <-> price_per_item                 r=0.933
  first_payment_value            <-> first_avg_price                r=0.913
  first_payment_value            <-> price_per_item                 r=0.913
  first_payment_value            <-> first_pymt_credit_card         r=0.848
  first_pymt_credit_card         <-> first_total_price              r=0.839
  first_total_freight            <-> first_avg_freight              r=0.801
  first_pymt_credit_card         <-> first_avg_price                r=0.8
  first_pymt_credit_card         <-> price_per_item                 r=0.8


In [11]:
# Drop correlated features
drop_cols = [
    'price_per_item',        # r=1.0 with first_avg_price
    'first_total_price',     # r=0.985 with first_payment_value
    'first_avg_price',       # r=0.913 with first_payment_value
    'first_avg_freight',     # r=0.801 with first_total_freight
    'lifetime_orders',       # leakage — this IS the target
]

user_df = user_df.drop(columns=[c for c in drop_cols if c in user_df.columns])

print(f"Shape after drops: {user_df.shape}")

# Save
user_df.to_csv('../processed/loyalty_modelling_dataset.csv', index=False)
print(f"Saved loyalty_modelling_dataset.csv")
print(f"Positive rate: {user_df['y_loyalty'].mean()*100:.2f}%")

Shape after drops: (96096, 38)
Saved loyalty_modelling_dataset.csv
Positive rate: 3.12%
